# Reproduce ChakraTS-OS-10M on GIFT-Eval

Reruns [GIFT-Eval](https://huggingface.co/spaces/Salesforce/GIFT-Eval) for [ChakraTS-OS-10M](https://huggingface.co/yhatlabs/ChakraTS-OS-10M) with the benchmark's own evaluation code, then compares
each result with the submitted one in this folder.

- `MODE = "quick"` (default): 13 small configurations, a few minutes; `MODE = "task"` reruns one configuration.
- `MODE = "full"`: all 97 configurations. Use a GPU: about 59 minutes on one NVIDIA H200.

The model runs on your own machine with [yhatmodels](https://github.com/yhatlabs/yhatmodels). Its first download asks you to accept the model license once with a free yhatlabs.com account; in Colab or on a server, set `YHAT_MODELS_TOKEN` (copy it from https://yhatlabs.com/models/license).

In [ ]:
# Setup: the gift-eval repository (its data loader and dataset properties) and the data. This notebook then runs from
# gift-eval's notebooks/ folder, as the original does; reruns are written to rerun/ next to this notebook.
import os, subprocess, sys, urllib.request
HERE = os.getcwd()
GIFT_COMMIT = "1b520f911c24e3f45d990a6dc966fadab2ba475b"
if not os.path.exists("gift-eval"):
    subprocess.run(["git", "clone", "-q", "https://github.com/SalesforceAIResearch/gift-eval"], check=True)
subprocess.run(["git", "-C", "gift-eval", "checkout", "-q", GIFT_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "./gift-eval", "pyarrow<26", "yhatmodels"], check=True)
if not os.path.exists("all_results.csv"):  # in Colab: the submitted file from GitHub
    urllib.request.urlretrieve("https://raw.githubusercontent.com/yhatlabs/yhatlabs-cookbook/main/benchmarks/gift_eval/ChakraTS-OS-10M/all_results.csv", "all_results.csv")
from huggingface_hub import snapshot_download
os.environ["GIFT_EVAL"] = snapshot_download("Salesforce/GiftEval", repo_type="dataset")
os.chdir("gift-eval/notebooks")

In [ ]:
import os

MODEL = "ChakraTS-OS-10M"   # short name, Hugging Face repository or local folder
MODEL_NAME = "ChakraTS-OS-10M"
DEVICE = None               # None: NVIDIA GPU, else Apple GPU, else CPU

MODE = "quick"              # "quick" (13 small configurations) | "task" (TASK below) | "full" (all 97)
TASK = "us_births/D/short"
SUBMITTED = os.path.join(HERE, "all_results.csv")  # the submitted results, printed next to each rerun


In [ ]:
import json, csv, time
import numpy as np, pandas as pd
from gift_eval.data import Dataset
from gluonts.model.forecast import QuantileForecast
from gluonts.model import evaluate_forecasts
from gluonts.ev.metrics import MSE, MAE, MASE, MAPE, SMAPE, MSIS, RMSE, NRMSE, ND, MeanWeightedSumQuantileLoss
from gluonts.time_feature import get_seasonality

QUANTILES = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
QKEYS = [str(q) for q in QUANTILES]
METRICS = [MSE(forecast_type="mean"), MSE(forecast_type=0.5), MAE(), MASE(), MAPE(), SMAPE(), MSIS(), RMSE(), NRMSE(), ND(), MeanWeightedSumQuantileLoss(quantile_levels=QUANTILES)]
COLUMNS = ["dataset", "model", "eval_metrics/MSE[mean]", "eval_metrics/MSE[0.5]", "eval_metrics/MAE[0.5]", "eval_metrics/MASE[0.5]", "eval_metrics/MAPE[0.5]",
           "eval_metrics/sMAPE[0.5]", "eval_metrics/MSIS", "eval_metrics/RMSE[mean]", "eval_metrics/NRMSE[mean]", "eval_metrics/ND[0.5]",
           "eval_metrics/mean_weighted_sum_quantile_loss", "domain", "num_variates"]

# The 97 configurations, enumerated exactly as the repository's other notebooks do.
short_datasets = "m4_yearly m4_quarterly m4_monthly m4_weekly m4_daily m4_hourly electricity/15T electricity/H electricity/D electricity/W solar/10T solar/H solar/D solar/W hospital covid_deaths us_births/D us_births/M us_births/W saugeenday/D saugeenday/M saugeenday/W temperature_rain_with_missing kdd_cup_2018_with_missing/H kdd_cup_2018_with_missing/D car_parts_with_missing restaurant hierarchical_sales/D hierarchical_sales/W LOOP_SEATTLE/5T LOOP_SEATTLE/H LOOP_SEATTLE/D SZ_TAXI/15T SZ_TAXI/H M_DENSE/H M_DENSE/D ett1/15T ett1/H ett1/D ett1/W ett2/15T ett2/H ett2/D ett2/W jena_weather/10T jena_weather/H jena_weather/D bitbrains_fast_storage/5T bitbrains_fast_storage/H bitbrains_rnd/5T bitbrains_rnd/H bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
med_long_datasets = "electricity/15T electricity/H solar/10T solar/H kdd_cup_2018_with_missing/H LOOP_SEATTLE/5T LOOP_SEATTLE/H SZ_TAXI/15T M_DENSE/H ett1/15T ett1/H ett2/15T ett2/H jena_weather/10T jena_weather/H bitbrains_fast_storage/5T bitbrains_rnd/5T bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
pretty_names = {"saugeenday": "saugeen", "temperature_rain_with_missing": "temperature_rain", "kdd_cup_2018_with_missing": "kdd_cup_2018", "car_parts_with_missing": "car_parts"}
dataset_properties_map = json.load(open("dataset_properties.json"))

def config_of(ds_name, term):
    if "/" in ds_name:
        ds_key, ds_freq = ds_name.split("/"); ds_key = pretty_names.get(ds_key.lower(), ds_key.lower())
    else:
        ds_key = pretty_names.get(ds_name.lower(), ds_name.lower()); ds_freq = dataset_properties_map[ds_key]["frequency"]
    return f"{ds_key}/{ds_freq}/{term}", ds_key

ALL = []  # (ds_name, term, config, ds_key)
for ds_name in short_datasets.split():
    for term in ["short", "medium", "long"]:
        if term in ("medium", "long") and ds_name not in med_long_datasets.split(): continue
        cfg, key = config_of(ds_name, term); ALL.append((ds_name, term, cfg, key))
assert len(ALL) == 97

QUICK = ["us_births/D/short", "us_births/W/short", "us_births/M/short", "saugeen/M/short", "saugeen/W/short", "ett1/W/short", "ett1/D/short", "ett2/W/short", "ett2/D/short", "bizitobs_l2c/H/short", "bizitobs_l2c/H/long", "jena_weather/D/short", "m_dense/D/short"]
SELECTED = [x for x in ALL if (MODE == "full") or (MODE == "task" and x[2] == TASK) or (MODE == "quick" and x[2] in QUICK)]
print(f"{len(SELECTED)} configuration(s) selected")

In [ ]:
from yhatmodels import ChakraTS

model = ChakraTS.from_pretrained(MODEL, device=DEVICE)

def windows(entries):
    # an item's test windows are yielded oldest first: index j = the j-th window of that item
    seen, out = {}, {}
    for idx, e in enumerate(entries):
        j = seen.get(e["item_id"], 0); seen[e["item_id"]] = j + 1; out.setdefault(j, []).append(idx)
    return out

def forecasts_for(ds_name, term, dataset, multivariate):
    inputs = list(dataset.test_data.input); labels = list(dataset.test_data.label)
    # multivariate datasets: each series is forecast whole (all variates); gluonts scores it per variate
    source = list(Dataset(name=ds_name, term=term, to_univariate=False).test_data.input) if multivariate else inputs
    preset = "pro"  # the default: multi-scale up to horizon 400, one patch size beyond
    group = 16  # series without covariates are read 16 at a time when a window's series share a cutoff
    use_past = True  # past_feat_dynamic_real, whenever the dataset provides it
    q = {}  # (item_id as test_data names it, window) -> [horizon, 9]
    for j, idx in sorted(windows(source).items()):  # one window per call, never two
        targets, past = [], []
        for x in idx:
            e = source[x]
            targets.append(np.atleast_2d(np.asarray(e["target"], dtype=np.float32)))
            pc = e.get("past_feat_dynamic_real")
            past.append({f"cov{c}": r for c, r in enumerate(np.atleast_2d(np.asarray(pc, dtype=np.float32)))}
                        if use_past and pc is not None and np.size(pc) else {})
        # read together only if every series of this window is cut at the same moment
        aligned = len({str(source[x]["start"] + np.shape(source[x]["target"])[-1]) for x in idx}) == 1
        fc = model.forecast(targets, dataset.prediction_length, quantiles=QUANTILES, preset=preset, cross_series=group if aligned else 1,
                            past=past if any(past) else None)
        for x, f in zip(idx, fc):
            iid = str(source[x]["item_id"])
            if multivariate:
                for d in range(f.shape[0]): q[(f"{iid}_dim{d}", j)] = f[d]
            else:
                q[(iid, j)] = f[0]
    # evaluate_forecasts pairs forecasts with test instances by position
    seen, out = {}, []
    for x, e in enumerate(inputs):
        iid = str(e["item_id"]); j = seen.get(iid, 0); seen[iid] = j + 1
        out.append(QuantileForecast(forecast_arrays=np.ascontiguousarray(q[(iid, j)].T), start_date=labels[x]["start"], forecast_keys=QKEYS, item_id=e["item_id"]))
    return out, len(inputs)

In [ ]:
submitted = pd.read_csv(SUBMITTED).set_index("dataset") if os.path.exists(SUBMITTED) else None
out_dir = os.path.join(HERE, "rerun"); os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "all_results.csv" if MODE == "full" else f"rerun_{MODE}.csv")
with open(out_path, "w", newline="") as f: csv.writer(f).writerow(COLUMNS)

rows, n_forecasts, t_start = [], 0, time.monotonic()
for n, (ds_name, term, cfg, key) in enumerate(SELECTED):
    to_univariate = Dataset(name=ds_name, term=term, to_univariate=False).target_dim != 1
    dataset = Dataset(name=ds_name, term=term, to_univariate=to_univariate)
    t0 = time.monotonic(); fc, count = forecasts_for(ds_name, term, dataset, to_univariate); n_forecasts += count
    res = evaluate_forecasts(fc, test_data=dataset.test_data, metrics=METRICS, axis=None, batch_size=1024, mask_invalid_label=True,
                             allow_nan_forecast=False, seasonality=get_seasonality(dataset.freq))
    row = [cfg, MODEL_NAME] + [float(res[c.split("/")[1]].iloc[0]) for c in COLUMNS[2:13]] + [dataset_properties_map[key]["domain"], dataset_properties_map[key]["num_variates"]]
    rows.append(row)
    with open(out_path, "a", newline="") as f: csv.writer(f).writerow(row)
    mase, crps = row[5], row[12]; line = f"[{n+1}/{len(SELECTED)}] {cfg}: {count} forecasts in {time.monotonic()-t0:.0f}s | MASE {mase:.4f} CRPS {crps:.4f}"
    if submitted is not None and cfg in submitted.index:
        s = submitted.loc[cfg]; sm, sc = s["eval_metrics/MASE[0.5]"], s["eval_metrics/mean_weighted_sum_quantile_loss"]
        line += f" | submitted MASE {sm:.4f} ({(mase/sm-1)*100:+.1f}%) CRPS {sc:.4f} ({(crps/sc-1)*100:+.1f}%)"
    print(line, flush=True)
elapsed = time.monotonic() - t_start
print(f"wrote {out_path}: {len(rows)} rows, {n_forecasts} forecasts in {elapsed/60:.1f} min")